# RoBERTa
### RoBERTa improves BERT with new pretraining objectives, demonstrating BERT was undertrained and training design is important. The pretraining objectives include dynamic masking, sentence packing, larger batches and a byte-level BPE tokenizer.

1. Text Classification (text-classification)
This is the most common use case for RoBERTa. It can be used for tasks like sentiment analysis, spam detection, or any binary/multi-class classification task.


In [3]:
import torch
from transformers import pipeline

pipeline = pipeline(
    task="text-classification",
    model="FacebookAI/roberta-base",
    torch_dtype=torch.float16,
    device=0
)

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cpu


In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch.nn.functional as F

# Loading tokenizer and classification model
tokenizer = AutoTokenizer.from_pretrained("FacebookAI/roberta-base")
model = AutoModelForSequenceClassification.from_pretrained(
    "FacebookAI/roberta-base",  # <- we need the insert fine-tuned model checkpoint here
    num_labels=2,               # <- adjust based on our classification task
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa"
)

# Sample text 
text = "This code contains a buffer overflow vulnerability."
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to("cuda")

# Forward pass
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    probs = F.softmax(logits, dim=-1)
    predicted_class = torch.argmax(probs, dim=-1).item()

print(f"Predicted class: {predicted_class}, Probabilities: {probs}")


Note: you may need to restart the kernel to use updated packages.


ValueError: Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`

<!-- 3. Sequence-to-Sequence (text2text-generation)
If you're building a system that fixes vulnerabilities (e.g., converts vulnerable code to secure code), then text2text-generation would be a better fit. -->

2.Sequence-to-Sequence (text2text-generation)
This is useful for fixing vulnerabilities, where the model generates a secure version of the code from a vulnerable one. It can also be used for tasks like summarization or translation.

In [2]:
import torch
from transformers import pipeline

pipeline = pipeline(
    task="text2text-generation",
    model="Salesforce/codet5-small",
    torch_dtype=torch.float16,
    device=0
)


/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cpu


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load tokenizer and seq2seq model
tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5-small")
model = AutoModelForSeq2SeqLM.from_pretrained(
    "Salesforce/codet5-small", # <- we need the insert fine-tuned model checkpoint here
    torch_dtype=torch.float16,
    device_map="auto"
)

# Input text: vulnerable code or prompt
text = "Fix the vulnerability: char *buffer = malloc(10); strcpy(buffer, user_input);"

# Tokenize input
inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to("cuda")

# Generate output
with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_length=128,
        num_beams=4,
        early_stopping=True
    )

# Decode the output
output_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(f"Generated output: {output_text}")


AssertionError: Torch not compiled with CUDA enabled